# 3장 — 손실의 직선: 스케일링 법칙 📉

웹 랩의 미니 LM 실험을 코드로 재현합니다 — **신의 시점**(진짜 전이표를 아는) 8단어
"고양이 세계" 언어에 데이터를 10배씩 늘려 먹이며 손실을 실측하고,

1. 총 손실이 **바닥**(줄일 수 없는 손실)에 달라붙으며 휘는 것,
2. 바닥을 뺀 **초과 손실**이 로그-로그에서 **직선**(멱법칙)이 되는 것,
3. 그 직선으로 **10배 큰 훈련을 예측**하는 것 (스케일링 법칙의 존재 이유)

까지 확인합니다. 난수는 웹 랩과 같은 mulberry32(시드 42) — 숫자가 전 자릿수 일치합니다.

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 여기서 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch3')

## 1. 고양이 세계 — 진짜 규칙과 바닥

전이 규칙: `고양이→는`, `는→(밥을 50% | 물을 30% | 잔다 20%)`, `밥을→(먹는다 90% | 마신다 10%)`,
`물을→(마신다 90% | 먹는다 10%)`, `동사→.`, `.→고양이`

진짜 전이표를 아니까 **완벽히 배워도 남는 불확실성** — 바닥 H — 를 직접 계산할 수 있습니다.

In [ ]:
import math
import numpy as np

M32 = 0xFFFFFFFF

def imul(a, b):
    return (a * b) & M32

def mulberry32(seed):
    """웹 랩과 동일한 난수 생성기 — 같은 시드면 같은 말뭉치."""
    a = seed & M32
    def rand():
        nonlocal a
        a = (a + 0x6D2B79F5) & M32
        t = a
        t = imul(t ^ (t >> 15), t | 1) & M32
        t = (t ^ ((t + imul(t ^ (t >> 7), t | 61)) & M32)) & M32
        return (t ^ (t >> 14)) / 4294967296
    return rand

V = 8
WORDS = ['고양이', '는', '밥을', '물을', '먹는다', '마신다', '잔다', '.']
P = [
    [0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0.5, 0.3, 0, 0, 0.2, 0],
    [0, 0, 0, 0, 0.9, 0.1, 0, 0],
    [0, 0, 0, 0, 0.1, 0.9, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0],
]

# 정상 분포 π (거듭제곱 반복) → 바닥 H = 진짜 조건부 엔트로피 (nat/토큰)
pi = [1 / V] * V
for _ in range(500):
    nxt = [0.0] * V
    for i in range(V):
        for j in range(V):
            nxt[j] += pi[i] * P[i][j]
    pi = nxt
H = sum(pi[i] * P[i][j] * -math.log(P[i][j]) for i in range(V) for j in range(V) if P[i][j] > 0)

print(f'바닥 H = {H:.6f} nat/토큰  → 최소 퍼플렉시티 e^H = {math.exp(H):.3f}')
assert abs(H - 0.268692) < 1e-6
assert abs(math.exp(H) - 1.308) < 1e-3, '완벽히 배워도 PPL 1.31 — 언어가 품은 갈래(5부 7장)'
print('✔ 완벽히 배워도 여기까지 — 바닥의 정체는 이 언어의 엔트로피(5부 3장)')

## 2. 데이터 스윕 — D를 10배씩 늘려 먹이며 손실 실측

바이그램 학습기(앞 단어별로 다음 단어를 **세는** 것 + add-α 평활)에 D 토큰을 먹이고,
진짜 분포 대비 교차 엔트로피(4부 11장의 그 손실)를 잽니다.

In [ ]:
def corpus(n, seed):
    rand = mulberry32(seed)
    out, s = [], 7
    for _ in range(n):
        r, acc = rand(), 0.0
        for j in range(V):
            acc += P[s][j]
            if r < acc:
                break
        out.append(j)
        s = j
    return out

ALPHA = 0.1

def loss_after(d, tokens):
    counts = [[0] * V for _ in range(V)]
    s = 7
    for k in range(d):
        counts[s][tokens[k]] += 1
        s = tokens[k]
    L = 0.0
    for i in range(V):
        row_sum = sum(counts[i])
        for j in range(V):
            if P[i][j] == 0:
                continue
            phat = (counts[i][j] + ALPHA) / (row_sum + ALPHA * V)
            L += pi[i] * P[i][j] * -math.log(phat)
    return L

DS = [100, 300, 1000, 3000, 10000, 30000, 100000]
tokens = corpus(100000, 42)
rows = [(d, loss_after(d, tokens)) for d in DS]

print('      D |  손실 L   | 초과 L−H')
print('-' * 38)
for d, L in rows:
    print(f'{d:>7} | {L:.6f} | {L - H:.3e}')

# 웹 랩·본문 수치 못박기
by = dict(rows)
assert abs(by[100] - 0.348807) < 1e-6
assert abs(by[1000] - 0.277965) < 1e-6
assert abs(by[100000] - 0.268749) < 1e-6
print()
print('✔ 웹 랩과 전 자릿수 일치 — 내려가되, 바닥(0.2687)에 달라붙는다')

## 3. 종이를 바꿔 보기 — 바닥을 빼면 직선이 된다

왼쪽: 총 손실(휘면서 바닥에 붙음). 오른쪽: 초과 손실의 로그-로그 — `np.polyfit` 으로
기울기를 재면(2장에서 배운 "직선에서 법칙 읽기") 멱법칙이 확인됩니다.

In [ ]:
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False
except Exception:
    pass

ds = np.array([d for d, _ in rows], dtype=float)
Ls = np.array([L for _, L in rows])
ex = Ls - H

b_fit, a_fit = np.polyfit(np.log10(ds), np.log10(ex), 1)
r = np.corrcoef(np.log10(ds), np.log10(ex))[0, 1]
r2 = r * r

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.9))
ax1.semilogx(ds, Ls, 'o-', color='#6366f1')
ax1.axhline(H, color='#f59e0b', ls='--', label=f'바닥 H = {H:.3f}')
ax1.set_title('총 손실 — 휘면서 바닥에 붙는다')
ax1.set_xlabel('데이터 D (토큰)'); ax1.set_ylabel('손실 (nat)')
ax1.legend()

ax2.loglog(ds, ex, 'o', color='#4f46e5')
fit_y = 10 ** (a_fit + b_fit * np.log10(ds))
ax2.loglog(ds, fit_y, '-', color='#10b981', label=f'기울기 {b_fit:.2f} (R²={r2:.3f})')
ax2.set_title('초과 손실 (L − H) — 로그-로그에서 직선!')
ax2.set_xlabel('데이터 D (토큰)'); ax2.set_ylabel('초과 손실')
ax2.legend()
plt.tight_layout()
plt.show()

assert abs(b_fit - (-1.062)) < 0.01, '기울기 ≈ −1.06 — 데이터 10배당 남은 배울 것 약 1/10'
assert r2 > 0.995
print(f'✔ 스케일링 법칙 실측: 초과 손실 ∝ D^({b_fit:.2f}),  R² = {r2:.4f}')
print('   → L(D) = E + c·D^(−b) — 바닥 + 멱법칙')

## 4. 예측 게임 — 싼 실험 5개로 10배 큰 훈련 맞히기

**앞 5점(D=100~10,000)만으로** 직선을 긋고 D=100,000까지 연장(외삽)합니다.
실제 프런티어 훈련이 수천 배 작은 실험들로 최종 손실을 예측하는 것과 같은 원리예요.

In [ ]:
b5, a5 = np.polyfit(np.log10(ds[:5]), np.log10(ex[:5]), 1)
pred_ex = 10 ** (a5 + b5 * 5)          # log10(100,000) = 5
pred_L = H + pred_ex
real_L = by[100000]
err = abs(pred_L - real_L) / real_L * 100

print(f'앞 5점 적합: 기울기 {b5:.3f}')
print(f'D=100,000 예측 손실 = {pred_L:.4f}  vs  실측 = {real_L:.4f}  (오차 {err:.3f}%)')
assert err < 0.05, '오차 0.05% 미만 — 작은 실험이 큰 훈련을 맞혔다'
print()
print('✔ 이것이 스케일링 법칙의 존재 이유 — 지르기 전에 예측한다')

## 정리 — 오늘 가져갈 세 문장

1. 학습 곡선의 정체는 **바닥 + 멱법칙**: L(D) = E + c·D⁻ᵇ (E = 언어의 엔트로피).
2. 바닥을 뺀 **초과 손실이 로그-로그의 직선** — 우리 toy는 기울기 ≈ −1, 진짜 LLM은
   −0.05~−0.3 수준으로 훨씬 완만(그래서 10배를 부어도 조금 내려간다).
3. 직선은 **연장할 수 있다** — 싼 실험으로 큰 훈련을 예측(오차 0.01% 수준)하는 것이
   프런티어의 표준 관행.

다음 **4장 — 몸집과 밥의 황금비(Chinchilla)**: N축·D축 법칙을 합치고 예산 C = 6ND를
고정하면, "크게 잠깐 vs 작게 오래"의 최적 배분이 계산으로 나옵니다.